# Machine Learning Training Pipeline

**Model Pipelines, Training and Evaluation Scripts**

Exact reproduction of ablation_retrain.py

## 1. Library Imports

In [ ]:
import os
import warnings
import joblib
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, ShuffleSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Libraries imported')

## 2. Configuration

In [ ]:
RANDOM_STATE = 52
N_TRIALS = 300
N_CORES = 16
N_CV_OPT = N_CORES * 2

SS_OPT = ShuffleSplit(n_splits=N_CV_OPT, test_size=0.20, random_state=RANDOM_STATE)
print(f'Config: {N_CV_OPT}-split CV, {N_TRIALS} trials, seed={RANDOM_STATE}')

## 3. Data Loading

In [ ]:
df = pd.read_excel(r'C:\Users\HWY\Desktop\Chen\水热炭化-CDs\CDs20260413.xlsx')

X = df.drop('QY', axis=1)
y = df['QY']

_categories = list(dict.fromkeys(X['Type']))
_type_mapping = {cat: idx + 1 for idx, cat in enumerate(_categories)}
X = X.copy()
X['Type'] = X['Type'].map(_type_mapping)

print('='*60)
print(f'Samples: {len(df)} | Features: {X.shape[1]}')
print(f'Type mapping: {_type_mapping}')
print('='*60)

## 4. Train/Test Split + Imputation

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

slr_median_train = X_train_val['SLR'].median()
X_train_val = X_train_val.copy()
X_test = X_test.copy()
X_train_val['SLR'] = X_train_val['SLR'].fillna(slr_median_train)
X_test['SLR'] = X_test['SLR'].fillna(slr_median_train)

print(f'Train: {len(X_train_val)} | Test: {len(X_test)}')
print(f'SLR median (train only): {slr_median_train:.4f}')

## 5. Helper Functions

In [ ]:
def calc_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return (
        r2_score(y_true, y_pred),
        np.sqrt(mean_squared_error(y_true, y_pred)),
        mean_absolute_error(y_true, y_pred),
    )

def safe_cv_score(pipe, cv_n_jobs=-1, penalty_weight=0.2):
    try:
        scores = cross_val_score(
            pipe, X_train_val, y_train_val,
            cv=SS_OPT, scoring='r2', n_jobs=cv_n_jobs
        )
        if np.any(~np.isfinite(scores)):
            return float('-inf')
        return float(scores.mean() - penalty_weight * scores.std())
    except Exception:
        return float('-inf')

def run_cv_and_fit(make_pipe_fn):
    model = make_pipe_fn()
    model.fit(X_train_val, y_train_val)
    return model

def record_model(name, model, result_list, model_dict):
    tr2, tr_rmse, tr_mae = calc_metrics(y_train_val, model.predict(X_train_val))
    te2, te_rmse, te_mae = calc_metrics(y_test, model.predict(X_test))
    print(f'  Train  R2: {tr2:.4f} | RMSE: {tr_rmse:.4f} | MAE: {tr_mae:.4f}')
    print(f'  Test   R2: {te2:.4f} | RMSE: {te_rmse:.4f} | MAE: {te_mae:.4f}')
    model_dict[name] = model
    result_list.append({
        'Model': name,
        'Train_R2': round(tr2, 4), 'Train_RMSE': round(tr_rmse, 4), 'Train_MAE': round(tr_mae, 4),
        'Test_R2': round(te2, 4), 'Test_RMSE': round(te_rmse, 4), 'Test_MAE': round(te_mae, 4),
    })

def build_study(default_trial_params):
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE, multivariate=True, n_startup_trials=30,
        ),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=25)
    )
    study.enqueue_trial(default_trial_params)
    return study

print('Helper functions defined')

## 6. GBDT Training with Optuna

In [ ]:
all_results_tuned = []
trained_models_tuned = {}

print('\n' + '='*60)
print(f'Optuna Optimization (GBDT, {N_TRIALS} trials)')
print(f'Objective: {N_CV_OPT}-split ShuffleSplit')
print('Sampler: Multivariate TPE | Objective: mean - 0.2*std')
print('='*60)

print('\n[Optuna] GBDT')
_GBDT_DEFAULT = {
    'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3,
    'min_samples_split': 2, 'min_samples_leaf': 1,
    'subsample': 1.0, 'max_features': 1.0,
}

def objective_gbdt(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 1000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.001, 0.5, log=True),
        'max_depth':         trial.suggest_int('max_depth', 2, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 20),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'max_features':      trial.suggest_float('max_features', 0.3, 1.0),
    }
    pipe = Pipeline([('pre', StandardScaler()),
                     ('reg', GradientBoostingRegressor(**params, random_state=RANDOM_STATE))])
    return safe_cv_score(pipe, cv_n_jobs=-1)

study_gbdt = build_study(_GBDT_DEFAULT)
study_gbdt.optimize(objective_gbdt, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=True)
if study_gbdt.best_value < study_gbdt.trials[0].value - 1e-9:
    study_gbdt.best_params.update(_GBDT_DEFAULT)
print(f'Best objective: {study_gbdt.best_value:.4f} | Params: {study_gbdt.best_params}')

gbdt_model = run_cv_and_fit(
    lambda: Pipeline([('pre', StandardScaler()),
                      ('reg', GradientBoostingRegressor(**study_gbdt.best_params,
                                                        random_state=RANDOM_STATE))])
)
record_model('GBDT', gbdt_model, all_results_tuned, trained_models_tuned)

## 7. Results

In [ ]:
results_df = pd.DataFrame(all_results_tuned)
results_df = results_df.sort_values('Test_R2', ascending=False).reset_index(drop=True)

print('\n' + '='*60)
print('Model Performance (sorted by Test R2)')
print('='*60)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]['Model']
print(f'\nBest model: {best_model_name}')
print(f'   Test R2:   {results_df.iloc[0]["Test_R2"]:.4f}')
print(f'   Test RMSE: {results_df.iloc[0]["Test_RMSE"]:.4f}')

## 8. Save Models

In [ ]:
OUTPUT_DIR = r'C:\Users\HWY\Desktop\Chen\水热炭化-CDs\CDs-模型输出'
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_model = trained_models_tuned[best_model_name]
model_path = os.path.join(OUTPUT_DIR, f'CDs_QY_{best_model_name}_best_model.pkl')
joblib.dump(best_model, model_path)
print(f'Model saved: {model_path}')

results_path = os.path.join(OUTPUT_DIR, 'CDs_QY_comparison.csv')
results_df.to_csv(results_path, index=False, encoding='utf-8-sig')
print(f'Results saved: {results_path}')